## Bivariate VEC Experiments

This section contains the corresponding simulation procedures and forecasting experiments.
Users can either generate new synthetic datasets by selecting the simulation parameters or load existing datasets stored in the Data/VEC folder. The provided datasets correspond to the experiments conducted in the thesis and facilitate the replication of the reported results.

### Import Required Libraries

Run the Required Libraries before executing any simulation or forecasting experiment.

In [ ]:
import os
os.chdir("xxxx")
from model.Base import Base
from model.MSVR import MSVR
from model.utility import (
    create_dataset,
    create_dataset_antes,
    rmse,
    CustomMSVR,
    create_dataset_rez,
    rezago_sig
)

from scikeras.wrappers import KerasRegressor

from scipy.linalg import orth
from scipy.stats import multivariate_normal

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit,
    train_test_split
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank

import csv
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### Option 1: Generate Synthetic Data and Run the forecasting experiment
This option generates a synthetic bivariate VEC(1) process. The user can modify the model parameters, sample size, forecasting horizon, and the number of simulation replications. Before running the simulation, the user can define the following parameters:


| Parameter | Description |
|-----------|-------------|
| t | Length of the generated time series. |
| k | Dimension of the multivariate process. |
| p | Number of lags in the VEC model. |
| h | Forecast horizon (steps ahead). |
| c | Number of Cointegration relationships. |


Example:

t = 1000  # Length of the series \
k = 2     # Dimension of the vector Y \
p = 1     # Number of lags \
h = 1     # Forecast horizon \
c = 1   # Number of Cointegration relationships



### Functions for generating cointegrated time Series

In [2]:
def generate_phi(n, h):
    A = orth(np.random.randn(n, n))[:h, :]
    Phi = np.eye(n)
    if h == 1:
        Phi -= np.outer(A, A)
    elif h > 1:
        Phi -= A.T @ A
    return Phi

def cointegrated_vector(tt, n, h, C, Phi, burn=1000):
    """
    Simulate cointegrated time series data.
    Parameters:
    tt : int Time points to simulate.
    n : int Number of time series.
    h : int Number of cointegrating relations.
    C : array_like Covariance matrix for the innovations.
    burn : int, optional Number of initial points to discard (burn-in period)..
    """
    tot_t = tt + burn
    innov = multivariate_normal.rvs(mean=np.zeros(n),cov=C, size=tot_t)
    X = np.zeros((1 + tot_t, n))
    print("Phi:\n", Phi)
    for i in range(tot_t):
        X[i + 1, :] = Phi @ X[i, :] + innov[i, :]
    return X[burn + 1:, :]

In [ ]:

t = [50,200,500,1000,5000]  #  Length of the series
k = 2     # Dimension of the vector Y 
p = 1     # Number of lags 
h = 1     # Forecast horizon 
c = 1   # Number of Cointegration relationships
Phi = generate_phi(k, c)
import warnings
warnings.filterwarnings("ignore")

train_RMSE_var_dif = {size: [] for size in t}
test_RMSE_var_dif = {size: [] for size in t}
tiempo_var_dif = {size: [] for size in t}
train_RMSE_vec = {size: [] for size in t}
test_RMSE_vec = {size: [] for size in t}
tiempo_vec = {size: [] for size in t}
hiperparametros_svr = {size: [] for size in t}
vectores_soporte = {size: [] for size in t}
train_RMSE_svr = {size: [] for size in t}
test_RMSE_svr = {size: [] for size in t}
tiempo_msvr = {size: [] for size in t}
hiperparametros_svr_tscv = {size: [] for size in t}
vectores_soporte_tscv = {size: [] for size in t}
train_RMSE_svr_tscv = {size: [] for size in t}
test_RMSE_svr_tscv = {size: [] for size in t}
tiempo_msvr_tscv = {size: [] for size in t}


for size in t:
      a=size
      for no in range(100):
            print(f"-------------------------Size {a}-------------------------")
            print(f"-------------------------Iteration {no}--------------------------")

        # Generate series
            series = np.zeros((k, size))
            series = cointegrated_vector(size, k, no, np.eye(k), Phi)
            series = pd.DataFrame(series, columns=['Y1', 'Y2'])
            
        # --------------------------------
        # Forecasting experiment
        # --------------------------------
            train_size = int(len(series) * 0.7)
            train, test = series.iloc[:train_size], series.iloc[train_size:]
            test = test.reset_index(drop=True)

            # Fitting  VAR DIF model
            series_dif = series.diff().dropna()
            train_size_dif = int(len(series_dif) * 0.7)
            train_dif, test_dif = series_dif.iloc[:train_size_dif], series_dif.iloc[train_size_dif:]
            test_dif = test_dif.reset_index(drop=True) 
            model_var_dif = VAR(train_dif)
            results_var_dif = model_var_dif.fit(maxlags=1)
            lag_order = results_var_dif.k_ar
        
            modelo_var_train_dif = []
            modelo_var_test_dif = []
            #Train pred
            train_pred_dif = results_var_dif.fittedvalues
            # Test pred
            test_pred_dif=[]
            input_data = train_dif.values[-p:]
            for i in range(len(test_dif)):
                pred = results_var_dif.forecast(y=input_data, steps=h)
                test_pred_dif.append(pred[0])
                print
                input_data = np.vstack([input_data[1:], test_dif.values[i:i+1]])

            test_pred_dif = pd.DataFrame(test_pred_dif, columns=['Y1', 'Y2'])
            last_value = train.iloc[-1, :]
            train_pred_levels = pd.DataFrame(train_pred_dif.cumsum() + train.iloc[lag_order], columns=['Y1', 'Y2'])
            test_pred_levels = pd.DataFrame(test_pred_dif.cumsum() + last_value.values, columns=['Y1', 'Y2'])
            train_rmse_var_dif = rmse(train.iloc[lag_order + 1:].values, train_pred_levels)
            test_rmse_var_dif = rmse(test.values, test_pred_levels)
            train_RMSE_var_dif[size].append(train_rmse_var_dif)
            test_RMSE_var_dif[size].append(test_rmse_var_dif)

            # Fitting  VEC model
            vecm = VECM(train, k_ar_diff=0, coint_rank=c, deterministic="ci")
            vecm_fit = vecm.fit()
        
            #Train pred
            train_pred = vecm_fit.fittedvalues
            #Predicciones para test
            test_pred = []

            for i in range(len(test)):

                vecm_temp = VECM(
                    pd.concat([train, test.iloc[:i]]),
                    k_ar_diff=0,
                    coint_rank=c,
                    deterministic="ci"
                )

                vecm_temp_fit = vecm_temp.fit()

                pred = vecm_temp_fit.predict(steps=1)

                test_pred.append(pred[0])

            test_pred = np.array(test_pred)
              
            train_rmse_vec = np.sqrt(mean_squared_error(train.values[p:], train_pred))
            test_rmse_vec = np.sqrt(mean_squared_error(test.values, test_pred))
            train_RMSE_vec[size].append(train_rmse_vec)
            test_RMSE_vec[size].append(test_rmse_vec)
            print("Finish fitting of VEC")
             # Fit the MSVR model
            start_time = time.time()
            fechas = pd.DataFrame(list(range(len(series))))
            total = pd.concat([fechas,series], axis=1).values
            dim=len(total)
            #Dataset construction
            data=Base(total)
            data= data.base
            #Create the supervised learning dataset
            dataset = create_dataset_rez(data,dim,h,k,p)
            X, Y = dataset[:, :(0 - h*2)], dataset[:, (0-h*2):]
            #Train-test split
            X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)
            #Feature standardization
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            scaler_X.fit(X_train)
            scaler_y.fit(y_train)
            X_train_nor = scaler_X.transform(X_train)
            X_test_nor = scaler_X.transform(X_test)
            y_train_nor = scaler_y.transform(y_train)
            y_test_nor = scaler_y.transform(y_test)
            pipe = Pipeline([
                ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
            ])
            hyperparameters = {
                #'MSVR__kernel': ['poly'],
                'MSVR__kernel': ['poly','rbf','linear'],
                'MSVR__degree': [2,5],
                #'MSVR__degree': [1],
                'MSVR__gamma': [0.5,1],
                'MSVR__coef0': [0.1,0.5,1],
                'MSVR__C': [5,9,11,13],
                'MSVR__epsilon':[1,2],
            }

            #Con tscv 
            tscv=TimeSeriesSplit(n_splits=5)
            bm_tscv = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=tscv, verbose=0, error_score='raise')
            best_model_tscv = bm_tscv.fit(X_train_nor, y_train_nor)
            best_params_tscv = bm_tscv.best_params_
            msvr_tscv = MSVR(kernel=bm_tscv.best_params_.get("MSVR__kernel"), gamma=bm_tscv.best_params_.get("MSVR__gamma"),
                            epsilon=bm_tscv.best_params_.get("MSVR__epsilon"), C=bm_tscv.best_params_.get("MSVR__C"),
                            degree=bm_tscv.best_params_.get("MSVR__degree"), coef0=bm_tscv.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr_tscv.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor_tscv = msvr_tscv.predict(X_train_nor)
            testPred_svr_nor_tscv = msvr_tscv.predict(X_test_nor)
            trainPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor_tscv))
            testPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor_tscv))
            train_rmse_svr_tscv = rmse(y_train, trainPred_svr_tscv)
            test_rmse_svr_tscv = rmse(y_test, testPred_svr_tscv)
            hiperparametros_svr_tscv[size].append(best_params_tscv)
            vectores_soporte_tscv[size].append(msvr_tscv.NSV)
            train_RMSE_svr_tscv[size].append(train_rmse_svr_tscv)
            test_RMSE_svr_tscv[size].append(test_rmse_svr_tscv)
            end_time = time.time()
            execution_time_tscv = end_time - start_time
            tiempo_msvr_tscv[size].append(end_time - start_time) 

            #Sin tscv 
            bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise', random_state=42)
            best_model = bm.fit(X_train_nor, y_train_nor)
            best_params = bm.best_params_
            msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
                            epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
                            degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor = msvr.predict(X_train_nor)
            testPred_svr_nor = msvr.predict(X_test_nor)
            trainPred_svr  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor))
            testPred_svr  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor))
            train_rmse_svr = rmse(y_train, trainPred_svr)
            #print(train_rmse_svr)
            test_rmse_svr = rmse(y_test, testPred_svr)
            hiperparametros_svr[size].append(best_params)
            vectores_soporte[size].append(msvr.NSV)
            train_RMSE_svr[size].append(train_rmse_svr)
            test_RMSE_svr[size].append(test_rmse_svr)
            end_time = time.time()
            execution_time = end_time - start_time
            tiempo_msvr[size].append(end_time - start_time)
            print("VAR DIF Train RMSE:", train_rmse_var_dif, "Test RMS ", test_rmse_var_dif)
            print("VEC Train RMSE:", train_rmse_vec, "Test RMS ", test_rmse_vec)
            print("MSVR Train RMSE:", train_rmse_svr, "Test RMS ", test_rmse_svr)
            print("MSVR_tscv Train RMSE:", train_rmse_svr_tscv, "MSVR_tscv ", test_rmse_svr_tscv)

filename = f'results_ VEC_Bivariate_data_sim.csv'
with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Tamaño', 'Modelo', 'Hiperparametros', 'Vectores_soporte', 'Train RMSE', 'Test RMSE', 'Tiempo'])
        for size in t:
            writer.writerow([f'Tamaño: {size}', '', '', '', '', '', ''])
            for i in range(100):
                writer.writerow(['SVR', hiperparametros_svr[size][i], vectores_soporte[size][i], train_RMSE_svr[size][i], test_RMSE_svr[size][i], tiempo_msvr[size][i]])
            for i in range(100):
                writer.writerow(['SVR_tscv', hiperparametros_svr_tscv[size][i], vectores_soporte_tscv[size][i], train_RMSE_svr_tscv[size][i], test_RMSE_svr_tscv[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VEC ', 'N/A', 'N/A', train_RMSE_vec[size][i], train_RMSE_vec[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VAR dif', 'N/A', 'N/A', train_RMSE_var_dif[size][i], train_RMSE_var_dif[size][i], 'N/A'])
                
            

        print(f'Results saved in {filename}')
        
 
 
        

### Option 2: Load Existing Data
Instead of generating new data, users can load datasets stored in the repository under the Data/VEC directory. These datasets correspond to the simulation scenarios used in the thesis and allow direct replication of the reported results.\
\
*Important:* When using an existing dataset, the time series generation step can be skipped. However, the parameters t (series length), k (dimension), p (number of lags), h (forecast horizon), and col (target variable) must still be defined, since they are used by the subsequent forecasting, model fitting, and evaluation procedures.

#### *Optional: Generate and Save Synthetic Datasets* 

You can generate and save the synthetic datasets if you would like to keep a backup of the generated series or reuse them in future experiments.

In [ ]:
t = [50,200,500,1000,5000]  #  Length of the series
k = 2     # Dimension of the vector Y 
p = 1     # Number of lags 
h = 1     # Forecast horizon 
c = 1   # Number of Cointegration relationships
Phi = generate_phi(k, c)

folder_path = r'xxx'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    for size in t:
        excel_filename = os.path.join(folder_path, f'dataset_size_{size}.xlsx')
        with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
            for no in range(100):
                print(f"-------------------------Size {size}-------------------------")
                print(f"-------------------------Iteration {no}--------------------------")
                series = np.zeros((k, size))
                series = cointegrated_vector(size, k, no, np.eye(k), Phi)
                series = pd.DataFrame(series, columns=['Y1', 'Y2'])
                sheet_name = f"Iter_{no}"
                series.to_excel(writer, sheet_name=sheet_name, index=False)


## Run the forecasting model

In [ ]:
t = [50,200,500,1000,5000]  #  Length of the series
k = 2     # Dimension of the vector Y 
p = 1     # Number of lags 
h = 1     # Forecast horizon 
c = 1   # Number of Cointegration relationships
Phi = generate_phi(k, c)

import warnings
warnings.filterwarnings("ignore")

train_RMSE_var_dif = {size: [] for size in t}
test_RMSE_var_dif = {size: [] for size in t}
tiempo_var_dif = {size: [] for size in t}
train_RMSE_vec = {size: [] for size in t}
test_RMSE_vec = {size: [] for size in t}
tiempo_vec = {size: [] for size in t}
hiperparametros_svr = {size: [] for size in t}
vectores_soporte = {size: [] for size in t}
train_RMSE_svr = {size: [] for size in t}
test_RMSE_svr = {size: [] for size in t}
tiempo_msvr = {size: [] for size in t}
hiperparametros_svr_tscv = {size: [] for size in t}
vectores_soporte_tscv = {size: [] for size in t}
train_RMSE_svr_tscv = {size: [] for size in t}
test_RMSE_svr_tscv = {size: [] for size in t}
tiempo_msvr_tscv = {size: [] for size in t}


for size in t:
      a=size
      for no in range(100):
            print(f"-------------------------Tamaño {a}-------------------------")
            print(f"-------------------------Iteration {no}--------------------------")

            # Read series
            series = pd.read_excel(rf"xxx\dataset_size_{size}.xlsx",sheet_name=f"Iter_{no}")
            series = pd.DataFrame(series, columns=['Y1', 'Y2'])
            
        # --------------------------------
        # Forecasting experiment
        # --------------------------------
            train_size = int(len(series) * 0.7)
            train, test = series.iloc[:train_size], series.iloc[train_size:]
            test = test.reset_index(drop=True)

            # Fitting  VAR DIF model
            series_dif = series.diff().dropna()
            train_size_dif = int(len(series_dif) * 0.7)
            train_dif, test_dif = series_dif.iloc[:train_size_dif], series_dif.iloc[train_size_dif:]
            test_dif = test_dif.reset_index(drop=True) 
            model_var_dif = VAR(train_dif)
            results_var_dif = model_var_dif.fit(maxlags=1)
            lag_order = results_var_dif.k_ar
        
            modelo_var_train_dif = []
            modelo_var_test_dif = []
            #Train pred
            train_pred_dif = results_var_dif.fittedvalues
            # Test pred
            test_pred_dif=[]
            input_data = train_dif.values[-p:]
            for i in range(len(test_dif)):
                pred = results_var_dif.forecast(y=input_data, steps=h)
                test_pred_dif.append(pred[0])
                print
                input_data = np.vstack([input_data[1:], test_dif.values[i:i+1]])

            test_pred_dif = pd.DataFrame(test_pred_dif, columns=['Y1', 'Y2'])
            last_value = train.iloc[-1, :]
            train_pred_levels = pd.DataFrame(train_pred_dif.cumsum() + train.iloc[lag_order], columns=['Y1', 'Y2'])
            test_pred_levels = pd.DataFrame(test_pred_dif.cumsum() + last_value.values, columns=['Y1', 'Y2'])
            train_rmse_var_dif = rmse(train.iloc[lag_order + 1:].values, train_pred_levels)
            test_rmse_var_dif = rmse(test.values, test_pred_levels)
            train_RMSE_var_dif[size].append(train_rmse_var_dif)
            test_RMSE_var_dif[size].append(test_rmse_var_dif)

            # Fitting  VEC model
            vecm = VECM(train, k_ar_diff=0, coint_rank=c, deterministic="ci")
            vecm_fit = vecm.fit()
        
            #Train pred
            train_pred = vecm_fit.fittedvalues
            #Predicciones para test
            test_pred = []

            for i in range(len(test)):

                vecm_temp = VECM(
                    pd.concat([train, test.iloc[:i]]),
                    k_ar_diff=0,
                    coint_rank=c,
                    deterministic="ci"
                )

                vecm_temp_fit = vecm_temp.fit()

                pred = vecm_temp_fit.predict(steps=1)

                test_pred.append(pred[0])

            test_pred = np.array(test_pred)
              
            train_rmse_vec = np.sqrt(mean_squared_error(train.values[p:], train_pred))
            test_rmse_vec = np.sqrt(mean_squared_error(test.values, test_pred))
            train_RMSE_vec[size].append(train_rmse_vec)
            test_RMSE_vec[size].append(test_rmse_vec)
            print("Termine de ajustar modelo VEC")
             # Fit the MSVR model
            start_time = time.time()
            fechas = pd.DataFrame(list(range(len(series))))
            total = pd.concat([fechas,series], axis=1).values
            dim=len(total)
            #Dataset construction
            data=Base(total)
            data= data.base
            #Create the supervised learning dataset
            dataset = create_dataset_rez(data,dim,h,k,p)
            X, Y = dataset[:, :(0 - h*2)], dataset[:, (0-h*2):]
            #Train-test split
            X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)
            #Feature standardization
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            scaler_X.fit(X_train)
            scaler_y.fit(y_train)
            X_train_nor = scaler_X.transform(X_train)
            X_test_nor = scaler_X.transform(X_test)
            y_train_nor = scaler_y.transform(y_train)
            y_test_nor = scaler_y.transform(y_test)
            pipe = Pipeline([
                ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
            ])
            hyperparameters = {
                #'MSVR__kernel': ['poly'],
                'MSVR__kernel': ['poly','rbf','linear'],
                'MSVR__degree': [2,5],
                #'MSVR__degree': [1],
                'MSVR__gamma': [0.5,1],
                'MSVR__coef0': [0.1,0.5,1],
                'MSVR__C': [5,9,11,13],
                'MSVR__epsilon':[1,2],
            }

            #Con tscv 
            tscv=TimeSeriesSplit(n_splits=5)
            bm_tscv = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=tscv, verbose=0, error_score='raise')
            best_model_tscv = bm_tscv.fit(X_train_nor, y_train_nor)
            best_params_tscv = bm_tscv.best_params_
            msvr_tscv = MSVR(kernel=bm_tscv.best_params_.get("MSVR__kernel"), gamma=bm_tscv.best_params_.get("MSVR__gamma"),
                            epsilon=bm_tscv.best_params_.get("MSVR__epsilon"), C=bm_tscv.best_params_.get("MSVR__C"),
                            degree=bm_tscv.best_params_.get("MSVR__degree"), coef0=bm_tscv.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr_tscv.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor_tscv = msvr_tscv.predict(X_train_nor)
            testPred_svr_nor_tscv = msvr_tscv.predict(X_test_nor)
            trainPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor_tscv))
            testPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor_tscv))
            train_rmse_svr_tscv = rmse(y_train, trainPred_svr_tscv)
            test_rmse_svr_tscv = rmse(y_test, testPred_svr_tscv)
            hiperparametros_svr_tscv[size].append(best_params_tscv)
            vectores_soporte_tscv[size].append(msvr_tscv.NSV)
            train_RMSE_svr_tscv[size].append(train_rmse_svr_tscv)
            test_RMSE_svr_tscv[size].append(test_rmse_svr_tscv)
            end_time = time.time()
            execution_time_tscv = end_time - start_time
            tiempo_msvr_tscv[size].append(end_time - start_time) 

            #Sin tscv 
            bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise', random_state=42)
            best_model = bm.fit(X_train_nor, y_train_nor)
            best_params = bm.best_params_
            msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
                            epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
                            degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor = msvr.predict(X_train_nor)
            testPred_svr_nor = msvr.predict(X_test_nor)
            trainPred_svr  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor))
            testPred_svr  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor))
            train_rmse_svr = rmse(y_train, trainPred_svr)
            #print(train_rmse_svr)
            test_rmse_svr = rmse(y_test, testPred_svr)
            hiperparametros_svr[size].append(best_params)
            vectores_soporte[size].append(msvr.NSV)
            train_RMSE_svr[size].append(train_rmse_svr)
            test_RMSE_svr[size].append(test_rmse_svr)
            end_time = time.time()
            execution_time = end_time - start_time
            tiempo_msvr[size].append(end_time - start_time)
            print("VAR DIF Train RMSE:", train_rmse_var_dif, "Test RMS ", test_rmse_var_dif)
            print("VEC Train RMSE:", train_rmse_vec, "Test RMS ", test_rmse_vec)
            print("MSVR Train RMSE:", train_rmse_svr, "Test RMS ", test_rmse_svr)
            print("MSVR_tscv Train RMSE:", train_rmse_svr_tscv, "MSVR_tscv ", test_rmse_svr_tscv)

filename = f'results_ VEC_Bivariate_2026_data1.csv'
with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Tamaño', 'Modelo', 'Hiperparametros', 'Vectores_soporte', 'Train RMSE', 'Test RMSE', 'Tiempo'])
        for size in t:
            writer.writerow([f'Tamaño: {size}', '', '', '', '', '', ''])
            for i in range(100):
                writer.writerow(['SVR', hiperparametros_svr[size][i], vectores_soporte[size][i], train_RMSE_svr[size][i], test_RMSE_svr[size][i], tiempo_msvr[size][i]])
            for i in range(100):
                writer.writerow(['SVR_tscv', hiperparametros_svr_tscv[size][i], vectores_soporte_tscv[size][i], train_RMSE_svr_tscv[size][i], test_RMSE_svr_tscv[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VEC ', 'N/A', 'N/A', train_RMSE_vec[size][i], train_RMSE_vec[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VAR dif', 'N/A', 'N/A', train_RMSE_var_dif[size][i], train_RMSE_var_dif[size][i], 'N/A'])
                
            

        print(f'Resultados guardados en {filename}')
        


-------------------------Size 50-------------------------
-------------------------Iteration 0--------------------------
Termine de ajustar modelo VEC
Last element of window_data matches the last element of data, breaking the loop.
Final dataset length: 49
VAR DIF Train RMSE: 1.52270142475115 Test RMS  4.956639510403313
VEC Train RMSE: 0.9181442936636532 Test RMS  0.971151223167509
MSVR Train RMSE: 0.8893731947783314 Test RMS  1.1140128863770367
MSVR_tscv Train RMSE: 0.8913682569117616 MSVR_tscv  1.1205417816977323
-------------------------Tamaño 50-------------------------
-------------------------Iteration 1--------------------------
Termine de ajustar modelo VEC
Last element of window_data matches the last element of data, breaking the loop.
Final dataset length: 49
VAR DIF Train RMSE: 2.224856663792881 Test RMS  1.8573915209357692
VEC Train RMSE: 0.8770390698941629 Test RMS  1.273096756076708
MSVR Train RMSE: 0.9566786084744401 Test RMS  1.6203745685362436
MSVR_tscv Train RMSE: 1.5